In [117]:
import langchain
import os
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
from pydantic import BaseModel, Field
from langchain.messages import HumanMessage,AIMessage
from langchain.chat_models import init_chat_model

In [118]:
load_dotenv("../.env")

True

In [119]:
llm=init_chat_model(
    model="openai/gpt-oss-120b",
    model_provider="groq",
    temperature=0
)


In [120]:
tavily_tool=TavilySearch(max_results=3, topic="finance")

In [121]:
class CompoundGrowthInput(BaseModel):
    initial_amount:float=Field(description="Starting initial amount")
    annual_addition:float=Field(description="Annual amount contribution")
    rate_percent:float=Field(description="Expected annual rate of return in percent (e.g. 7.5)")
    years:int=Field(description="Total investment horizon in years")

In [122]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool

class CompoundGrowthInput(BaseModel):
    initial_amount: float = Field(description="Starting investment amount")
    annual_addition: float = Field(description="Annual amount contributed")
    rate_percent: float = Field(description="Expected annual rate of return in percent (e.g., 12 for 12%)")
    years: int = Field(description="Total investment horizon in years")

@tool("compound_growth_calculator", args_schema=CompoundGrowthInput)
def compound_growth_calculator(initial_amount: float, annual_addition: float, rate_percent: float, years: int) -> str:
    """Calculates future investment balance and total interest earned."""
    r = float(rate_percent) / 100.0
    total = float(initial_amount)
    contributions = float(initial_amount)
    
    for _ in range(int(years)):
        total = (total + float(annual_addition)) * (1.0 + r)
        contributions += float(annual_addition)
        
    interest_earned = total - contributions
    return (
        f"Initial: ${initial_amount:,.2f} | "
        f"Total Contributions: ${contributions:,.2f} | "
        f"Projected Balance: ${total:,.2f} | "
        f"Total Gain: ${interest_earned:,.2f}"
    )

In [123]:
class ExportReportInput(BaseModel):
    ticker: str=Field(description="Stock ticker symbol (e.g. AAPL, NVDA)")
    content:str=Field(description="Detailed research analysis summar to save")


In [124]:
@tool
def export_financial_report(ticker: str, content:str)->str:
    """Saves the financial research report to a local markdown file. Requires human oversight."""
    filename=f"{ticker.upper()}_research_brief.md"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"# Financial Analysis: {ticker.upper()}\n\n{content}\n")
    return f"Report successfully .saved to {filename}"


In [125]:
tools=[tavily_tool, Compound_growtn_calculator, export_financial_report]

In [126]:
checkpointer=InMemorySaver()

In [127]:
summarizer=SummarizationMiddleware(
    model=llm,
    trigger=("messages", 8),
    keep=("messages", 4)
)

agent=create_agent(
    model=llm,
    tools=tools,
    checkpointer=checkpointer,
    middleware=[summarizer],
    interrupt_before=["tools"],
    system_prompt=(
        "You are an expert Wall Street financial analyst."
        "Use Tavily to look up recent earnings, catalysts, and SEC developments."
        "Use the compound calculator for return scenarios."
        "When the user requests an exported report, call report_financial_report."
        "When reporting revenue figures, always verify the exact fiscal quarter and year. Prioritize official investor relations announcements or SEC 10-Q releases. Do not mix multi-quarter totals or different fiscal years."
    )
)

In [128]:
thread_config={"configurable":{"thread_id":"user_session_4"}}

In [129]:
from langchain_core.messages import HumanMessage, AIMessage

# Use a new thread_id so previous stalled states don't carry over
thread_config = {"configurable": {"thread_id": "session_live_1"}}

def run_chat(query: str):
    print(f"\nUser: {query}")
    
    # 1. First invocation with the user's prompt
    state = agent.invoke(
        {"messages": [HumanMessage(content=query)]},
        config=thread_config
    )
    
    # 2. If it paused on a tool call, resume it automatically to run the tool
    while True:
        last_msg = state["messages"][-1]
        
        # If the last message is an AIMessage with pending tool_calls and no text yet
        if isinstance(last_msg, AIMessage) and getattr(last_msg, "tool_calls", None) and not last_msg.content:
            print(f"⚙️ Running tool: {last_msg.tool_calls[0]['name']}...")
            # Resuming execution runs the tool node and calls LLM again
            state = agent.invoke(None, config=thread_config)
        else:
            break

    # 3. Print the final answer
    final_text = None
    for msg in reversed(state["messages"]):
        if isinstance(msg, AIMessage) and msg.content:
            final_text = msg.content
            break
            
    if final_text:
        print(f"\nAssistant:\n{final_text}")
    else:
        print(f"\n[Still waiting]: {state['messages'][-1]}")
        
    return state

In [130]:
state = run_chat("What are the latest revenue numbers and growth drivers for NVIDIA (NVDA)?")


User: What are the latest revenue numbers and growth drivers for NVIDIA (NVDA)?


⚙️ Running tool: tavily_search...
⚙️ Running tool: tavily_search...
⚙️ Running tool: tavily_search...
⚙️ Running tool: tavily_search...
⚙️ Running tool: tavily_search...
⚙️ Running tool: tavily_search...

Assistant:
**NVIDIA (NVDA) – First Quarter FY 2027 (ended April 26 2026)**  

| Metric | Figure | YoY / QoQ Change |
|--------|--------|------------------|
| **Total revenue** | **$81.6 billion** | +85 % YoY, +20 % QoQ |
| **Data‑Center revenue** | **$75.2 billion** | +92 % YoY, +21 % QoQ |
| **Edge‑Computing revenue** | $6.4 billion | +29 % YoY |
| **GAAP net income** | $58.3 billion | +211 % YoY |
| **GAAP EPS** | $2.39 | – |
| **Non‑GAAP EPS** | $1.87 | – |
| **Dividend** | $0.25 per share (up from $0.01) – payable June 26 2026 | – |
| **Share‑repurchase** | $20 billion returned this quarter; $80 billion additional buy‑back authorization | – |

*Sources: NVIDIA press release (May 20 2026) and Yahoo Finance coverage of the same release*【3e0961-00】, 【f39472-02】.

---

### Primary Gro

In [132]:
state = run_chat("If I invest $10,000 in this stock and add $2,000/year at an estimated 12% return for 5 years, what would the portfolio be worth?")


User: If I invest $10,000 in this stock and add $2,000/year at an estimated 12% return for 5 years, what would the portfolio be worth?

Assistant:
**Future‑value projection (annual compounding, contributions at year‑end)**  

| Parameter | Value |
|-----------|-------|
| Initial investment (PV) | $10,000 |
| Annual contribution (PMT) | $2,000 (added at the end of each year) |
| Expected annual return (r) | 12 % |
| Investment horizon (n) | 5 years |

**Calculation**

\[
\text{FV} = PV\,(1+r)^{n} \;+\; PMT \times \frac{(1+r)^{n}-1}{r}
\]

\[
\begin{aligned}
PV\,(1+r)^{n} &= 10{,}000 \times (1.12)^{5} \approx 10{,}000 \times 1.7635 = \$17{,}635. \\
PMT \times \frac{(1+r)^{n}-1}{r} &= 2{,}000 \times \frac{1.7635-1}{0.12}
= 2{,}000 \times 6.3628 \approx \$12{,}726. \\
\text{Total FV} &= 17{,}635 + 12{,}726 \approx \boxed{\$30{,}361}
\end{aligned}
\]

**Result:** After 5 years, assuming a steady 12 % annual return and $2,000 added at the end of each year, a $10,000 initial investment would

In [133]:
state = run_chat("Export this analysis into a summary report for NVDA.")


User: Export this analysis into a summary report for NVDA.
⚙️ Running tool: export_financial_report...

Assistant:
Your NVIDIA (NVDA) summary report has been exported and saved as **NVDA_research_brief.md**. Let me know if you’d like any further analysis, visualizations, or additional sections added!


In [134]:
with open("NVDA_research_brief.md", "r", encoding="utf-8") as f:
    print(f.read())

# Financial Analysis: NVDA

# NVIDIA (NVDA) – Q1 FY2027 Summary Report

**Fiscal Quarter:** Q1 FY2027 (ended April 26 2026)

---

## 1. Revenue Overview
| Metric | Figure | YoY / QoQ Change |
|--------|--------|------------------|
| **Total Revenue** | **$81.6 billion** | +85 % YoY, +20 % QoQ |
| **Data‑Center Revenue** | **$75.2 billion** | +92 % YoY, +21 % QoQ |
| **Edge‑Computing Revenue** | $6.4 billion | +29 % YoY |
| **GAAP Net Income** | $58.3 billion | +211 % YoY |
| **GAAP EPS** | $2.39 | — |
| **Non‑GAAP EPS** | $1.87 | — |
| **Dividend** | $0.25 per share (up from $0.01) – payable June 26 2026 |
| **Share‑Repurchase** | $20 billion returned this quarter; $80 billion additional buy‑back authorization |

*Sources: NVIDIA FY2027 Q1 earnings release (May 20 2026) and corroborating coverage on Yahoo Finance.*

---

## 2. Primary Growth Drivers
| Driver | Explanation |
|--------|-------------|
| **Blackwell AI‑chip architecture** | First‑generation Blackwell GPUs deliver up to 2× 